# Build species → virus lookup table

Produces **`IEDB/species_virus_lookup.csv`**, a taxonomy-based classification of every source species in the working table as virus / non-virus. Used by notebook 2 (`2_iedb_working_table.ipynb`) to optionally restrict the analysis to **viruses only**.

## Why taxonomy, not keywords
A keyword filter (`name contains "virus"`) silently drops real viruses whose IEDB name has no such token — e.g. `SARS-CoV1`, `Other SARS` (together ~1,650+ class I peptides). The rigorous definition of "is a virus" is: **the organism's NCBI taxonomy lineage sits under `Viruses` (taxid 10239)**.

## Method
1. Take the distinct `species` values from the working table; extract the NCBITaxon ID from `species_iri`.
2. Query NCBI taxonomy (`Bio.Entrez`) for each taxon's lineage; flag virus iff `10239`/`Viruses` is in the lineage.
3. The handful of species without an NCBITaxon ID are resolved manually (see `MANUAL_VIRUS`).

Output columns: `species, taxid, scientific_name, species_iri, is_virus`. The table is auditable — inspect/correct it before relying on it.

In [ ]:
import re
import time
import pandas as pd
from Bio import Entrez

# NCBI asks for an email with every request; be polite (<=3 req/s without an API key).
Entrez.email = "dpouyaba@broadinstitute.org"

WORKING_TABLE = "IEDB/iedb_human_pathogen_working_table_v2.csv"
OUT           = "IEDB/species_virus_lookup.csv"
VIRUS_TAXID   = "10239"   # NCBI taxonomy id for the Viruses superkingdom
BATCH         = 200       # taxids per efetch call

# Species with no NCBITaxon IRI in the export but known to be viral.
MANUAL_VIRUS = {"SARS-CoV1", "Other SARS"}

## 1. Distinct species and their NCBITaxon IDs

In [ ]:
df = (pd.read_csv(WORKING_TABLE, low_memory=False, usecols=["species", "species_iri"])
        .drop_duplicates("species"))

def extract_taxid(iri):
    if isinstance(iri, str):
        m = re.search(r"NCBITaxon_(\d+)", iri)
        if m:
            return m.group(1)
    return None

df["taxid"] = df["species_iri"].map(extract_taxid)
ids = df["taxid"].dropna().tolist()
print(f"distinct species: {len(df)}")
print(f"with NCBITaxon id: {len(ids)} | without: {df['taxid'].isna().sum()}")
print("species without NCBITaxon id:", df.loc[df['taxid'].isna(), 'species'].tolist())

## 2. Resolve lineages from NCBI taxonomy

Batched `efetch` against `db=taxonomy`. A species is a virus iff `Viruses` (taxid `10239`) appears anywhere in its lineage.

In [ ]:
lineage = {}   # taxid -> (scientific_name, is_virus)
for i in range(0, len(ids), BATCH):
    batch = ids[i:i + BATCH]
    handle = Entrez.efetch(db="taxonomy", id=",".join(batch), retmode="xml")
    recs = Entrez.read(handle)
    handle.close()
    for r in recs:
        tid = r["TaxId"]
        lineage_tids = [tid] + [x["TaxId"] for x in r.get("LineageEx", [])]
        is_virus = (VIRUS_TAXID in lineage_tids) or ("Viruses" in r.get("Lineage", ""))
        lineage[tid] = (r["ScientificName"], is_virus)
    print(f"  resolved {min(i + BATCH, len(ids))}/{len(ids)}")
    time.sleep(0.4)   # stay under the rate limit

print(f"resolved {len(lineage)} taxa")

## 3. Assemble, apply manual overrides, save

In [ ]:
df["scientific_name"] = df["taxid"].map(lambda t: lineage.get(t, (None, None))[0])
df["is_virus"] = df["taxid"].map(lambda t: lineage.get(t, (None, False))[1]).astype(bool)

# species without an NCBITaxon id: classify from MANUAL_VIRUS
manual_mask = df["species"].isin(MANUAL_VIRUS)
df.loc[manual_mask, "is_virus"] = True
df.loc[manual_mask & df["scientific_name"].isna(), "scientific_name"] = "(manual override)"

lookup = df[["species", "taxid", "scientific_name", "species_iri", "is_virus"]].sort_values("species")
lookup.to_csv(OUT, index=False)
print(f"saved {OUT}  ({len(lookup)} species)")
print(f"viruses: {int(lookup['is_virus'].sum())} | non-viruses: {int((~lookup['is_virus']).sum())}")

## 4. Sanity checks

In [ ]:
# viruses a naive keyword filter would have MISSED (no viral token in the name)
kw = re.compile(r"virus|viridae|viral|phage|viroid", re.I)
missed = lookup[lookup["is_virus"] & ~lookup["species"].fillna("").str.contains(kw)]
print(f"viruses without a viral keyword (keyword filter would drop these): {len(missed)}")
print(missed[["species", "scientific_name"]].to_string(index=False))

# non-viruses that DO contain a viral keyword (would be wrongly kept by a keyword filter)
false_kw = lookup[~lookup["is_virus"] & lookup["species"].fillna("").str.contains(kw)]
print(f"\nnon-viruses with a viral keyword: {len(false_kw)}")
print(false_kw[["species", "scientific_name"]].to_string(index=False))

In [ ]:
print("sample of virus species:")
print(lookup[lookup["is_virus"]][["species", "scientific_name"]].head(20).to_string(index=False))